# Feature Set Comparison (A/B)
**Purpose:** justify the feature store design for Assignment 2 (design-decision slide).

| Feature set | Description |
|---|---|
| **A1 original** | Assignment 1 gold feature store: 103 features, clickstream mean / std / min / max |
| **A1 slim** | A2 gold feature store without borrowed features: clickstream mean only |
| **A2 (chosen)** | A1 slim + 9 loan type counts (sample 1) + 10 ratio features (samples 2, 3) |

**Same setup as the training pipeline** (`utils/model_train.py`): training date 2024-06-01, train/test = applications 2023-01..2023-10 (80/20 stratified, seed 88), OOT = 2023-11..2023-12, only labels observed by the training date, imputer + scaler fitted on train, Logistic Regression and XGBoost (same random search).

**Decision metric: OOT AUC** (same as model selection). Production AUC (2024-06..2025-01) is shown as *post-hoc* only: it uses labels that were not known at training time, so it is not used for the decision.

In [1]:
import os
import glob
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import xgboost as xgb

import warnings
warnings.filterwarnings("ignore")

## 1. Load gold tables

In [2]:
def load_parquet_folder(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.parquet")))
    return pd.concat([pd.read_parquet(f) for f in files])

fs_a2 = load_parquet_folder("datamart/gold/feature_store/")
fs_a1 = load_parquet_folder("../assignment_1/datamart/gold/feature_store/")
labels = load_parquet_folder("datamart/gold/label_store/")

print("A2 feature store:", fs_a2.shape, "| A1 feature store:", fs_a1.shape, "| labels:", labels.shape)

A2 feature store: (12500, 65) | A1 feature store: (12500, 106) | labels: (12500, 5)


## 2. Define feature sets

In [3]:
ID_COLS = ["Customer_ID", "loan_id", "snapshot_date", "label"]

BORROWED = [c for c in fs_a2.columns if c.startswith("loan_type_")] + [
    "num_fin_products", "emi_to_salary", "debt_to_salary", "repayment_ability",
    "loans_per_credit_item", "debt_to_income", "log_annual_income",
    "investment_rate", "inq_per_loan", "has_credit_limit_change",
]

feature_sets = {
    "A1 original": (fs_a1, [c for c in fs_a1.columns if c not in ID_COLS]),
    "A1 slim": (fs_a2, [c for c in fs_a2.columns if c not in ID_COLS + BORROWED]),
    "A2 (chosen)": (fs_a2, [c for c in fs_a2.columns if c not in ID_COLS]),
}

for name, (_, cols) in feature_sets.items():
    print(f"{name:12s}: {len(cols)} features")

A1 original : 103 features
A1 slim     : 43 features
A2 (chosen) : 62 features


## 3. Same time windows as the training pipeline

In [4]:
model_train_date = pd.Timestamp("2024-06-01")
train_test_start, train_test_end = pd.Timestamp("2023-01-01"), pd.Timestamp("2023-10-01")
oot_start, oot_end = pd.Timestamp("2023-11-01"), pd.Timestamp("2023-12-01")
prod_start, prod_end = pd.Timestamp("2024-06-01"), pd.Timestamp("2025-01-01")

# labels observed by training date (for train/test/OOT); all labels for post-hoc production check
labels_known = labels[pd.to_datetime(labels["snapshot_date"]) <= model_train_date][["loan_id", "label"]]
labels_all = labels[["loan_id", "label"]]

def period(df, start, end):
    d = pd.to_datetime(df["snapshot_date"])
    return df[(d >= start) & (d <= end)]

## 4. Train and evaluate each feature set

In [5]:
PARAM_DIST = {
    'n_estimators': [25, 50],
    'max_depth': [2, 3],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.6, 0.8],
    'colsample_bytree': [0.6, 0.8],
    'gamma': [0, 0.1],
    'min_child_weight': [1, 3, 5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

results = []
xgb_models = {}

for name, (fs, cols) in feature_sets.items():
    train_test = period(fs, train_test_start, train_test_end).merge(labels_known, on="loan_id")
    oot = period(fs, oot_start, oot_end).merge(labels_known, on="loan_id")
    prod = period(fs, prod_start, prod_end).merge(labels_all, on="loan_id")

    X_train, X_test, y_train, y_test = train_test_split(
        train_test[cols], train_test["label"], test_size=0.2,
        random_state=88, shuffle=True, stratify=train_test["label"])

    imputer = SimpleImputer(strategy="median").fit(X_train)
    scaler = StandardScaler().fit(imputer.transform(X_train))
    prep = lambda X: scaler.transform(imputer.transform(X))

    lr = LogisticRegression(max_iter=1000, random_state=88).fit(prep(X_train), y_train)
    search = RandomizedSearchCV(xgb.XGBClassifier(eval_metric="logloss", random_state=88), PARAM_DIST,
                                scoring="roc_auc", n_iter=50, cv=3, random_state=42, n_jobs=-1)
    search.fit(prep(X_train), y_train)
    xgb_models[name] = (search.best_estimator_, cols)

    for model_type, model in [("logistic_regression", lr), ("xgboost", search.best_estimator_)]:
        score = lambda X: model.predict_proba(prep(X))[:, 1]
        results.append({
            "feature_set": name,
            "n_features": len(cols),
            "model": model_type,
            "auc_train": roc_auc_score(y_train, score(X_train)),
            "auc_test": roc_auc_score(y_test, score(X_test)),
            "auc_oot": roc_auc_score(oot["label"], score(oot[cols])),
            "auc_prod_posthoc": roc_auc_score(prod["label"], score(prod[cols])),
        })
    print(f"{name}: train {len(X_train)} | test {len(X_test)} | oot {len(oot)} | prod {len(prod)}")

results_pdf = pd.DataFrame(results)
results_pdf["gini_oot"] = 2 * results_pdf["auc_oot"] - 1
results_pdf.round(3)

A1 original: train 3982 | test 996 | oot 980 | prod 4024


A1 slim: train 3982 | test 996 | oot 980 | prod 4024


A2 (chosen): train 3982 | test 996 | oot 980 | prod 4024


,feature_set,n_features,model,auc_train,auc_test,auc_oot,auc_prod_posthoc,gini_oot
0,A1 original,103,logistic_regression,0.855,0.833,0.870,0.786,0.740
1,A1 original,103,xgboost,0.908,0.863,0.887,0.804,0.775
2,A1 slim,43,logistic_regression,0.849,0.832,0.872,0.786,0.744
3,A1 slim,43,xgboost,0.902,0.858,0.881,0.808,0.761
4,A2 (chosen),62,logistic_regression,0.852,0.833,0.874,0.789,0.748
5,A2 (chosen),62,xgboost,0.899,0.849,0.881,0.810,0.762


## 5. Summary: best model per feature set (by OOT AUC)

In [6]:
best = results_pdf.loc[results_pdf.groupby("feature_set")["auc_oot"].idxmax()]
best = best.set_index("feature_set").loc[list(feature_sets.keys())]
best[["n_features", "model", "auc_test", "auc_oot", "gini_oot", "auc_prod_posthoc"]].round(3)

,n_features,model,auc_test,auc_oot,gini_oot,auc_prod_posthoc
feature_set,,,,,,
A1 original,103,xgboost,0.863,0.887,0.775,0.804
A1 slim,43,xgboost,0.858,0.881,0.761,0.808
A2 (chosen),62,xgboost,0.849,0.881,0.762,0.810


## 6. How much do the borrowed features contribute? (A2 XGBoost feature importance)

In [7]:
model, cols = xgb_models["A2 (chosen)"]
importance = pd.DataFrame({"feature": cols, "importance": model.feature_importances_})
importance["borrowed"] = importance["feature"].isin(BORROWED)
importance = importance.sort_values("importance", ascending=False).reset_index(drop=True)

print("share of total importance from borrowed features:", round(importance.loc[importance.borrowed, "importance"].sum(), 3))
importance.head(15).round(3)

share of total importance from borrowed features: 0.154


,feature,importance,borrowed
0,Outstanding_Debt,0.148,False
1,Interest_Rate,0.112,False
2,num_fin_products,0.062,True
3,Num_Bank_Accounts,0.037,False
4,fe_5_mean,0.034,False
5,fe_10_mean,0.033,False
6,Num_Credit_Card,0.029,False
7,fe_9_mean,0.028,False
8,Changed_Credit_Limit,0.027,False
9,Delay_from_due_date,0.026,False


## 7. Conclusion

| Feature set | Features | Best model | OOT AUC | Prod AUC (post-hoc) |
|---|---|---|---|---|
| A1 original | 103 | XGBoost | 0.887 | 0.804 |
| A1 slim | 43 | XGBoost | 0.881 | 0.808 |
| A2 (chosen) | 62 | XGBoost | 0.881 | 0.810 |

1. **Performance is statistically the same for all three sets.** OOT differences are at most 0.006 AUC, about half of one standard error (OOT has 980 loans, so the AUC standard error is roughly 0.011). Neither the borrowed features nor dropping clickstream std/min/max changes predictive power in a meaningful way.
2. **Dropping clickstream std/min/max (60 features) costs nothing**, and even with the 19 borrowed features added, A2 has **41 fewer features than A1 original** (62 vs 103). Lecture 7: use the minimal feature set with the same performance to reduce drift risk. Fewer clickstream features also means less exposure to the clickstream data source, which stopped for new applicants from 2024-07 (see monitoring).
3. **The borrowed features are used by the model.** They make up **15% of XGBoost's importance**, and `num_fin_products` (from sample 2) is the **3rd most important feature**. They are interpretable business ratios (debt-to-salary, number of financial products), which helps explain decisions in lending. They slightly improve production AUC post-hoc (0.810 vs 0.808 vs 0.804), but that difference is too small to count as evidence.
4. **Decision: keep A2.** It matches the best OOT performance, uses 40% fewer features than A1 original, and adds explainable credit-risk ratios. We do **not** claim the borrowed features improve accuracy.